# Notebook 01 — Exploratory Data Analysis

**Purpose:** Understand the Spotify dataset before modelling.

Run `python scripts/run_eda.py` first to generate `data/processed/` arrays.
This notebook is for **visualisation only** — no data is modified here.

In [ ]:
import sys
sys.path.insert(0, '..')   # allow src/ imports from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.config import CFG
from src.data.loader import load_dataset
from src.data.preprocessor import clean_data

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

## 1. Load & inspect

In [ ]:
df_raw = load_dataset(CFG.raw_data_path)
df_raw.head()

In [ ]:
print('Shape:', df_raw.shape)
print('\nData types:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())

In [ ]:
df = clean_data(df_raw)
df.describe()

## 2. Class balance

In [ ]:
genre_counts = df[CFG.target_column].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
genre_counts.plot(kind='bar', ax=ax, color=sns.color_palette('tab10', len(genre_counts)))
ax.set_title('Class Distribution (Playlist Genre)')
ax.set_xlabel('Genre')
ax.set_ylabel('Number of tracks')
ax.tick_params(axis='x', rotation=15)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(genre_counts)

## 3. Feature distributions per genre

In [ ]:
genres   = sorted(df[CFG.target_column].unique())
features = CFG.audio_features
palette  = dict(zip(genres, sns.color_palette('tab10', len(genres))))

n_cols = 3
n_rows = (len(features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(features):
    for genre in genres:
        subset = df[df[CFG.target_column] == genre][feat]
        axes[i].hist(subset, bins=30, alpha=0.4, label=genre, color=palette[genre], density=True)
    axes[i].set_title(feat)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Density')

# Hide unused axes
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

handles = [plt.Rectangle((0,0),1,1, color=palette[g], alpha=0.5) for g in genres]
fig.legend(handles, genres, loc='lower right', ncol=3, title='Genre')
fig.suptitle('Audio Feature Distributions by Genre', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation heatmap

In [ ]:
corr = df[features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))   # show only lower triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, square=True, ax=ax,
    linewidths=0.5,
)
ax.set_title('Audio Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight high correlations (|r| > 0.5)
print('\nHigh correlations (|r| > 0.5):')
corr_unstacked = corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool)).stack()
high_corr = corr_unstacked[corr_unstacked.abs() > 0.5]
print(high_corr.sort_values(ascending=False).to_string())

## 5. PCA explained variance curve

This guides the choice of `n_components` in `src/config.py`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X = df[features].values.astype(float)
X_scaled = MinMaxScaler(feature_range=(0, float(np.pi))).fit_transform(X)

pca_full = PCA(n_components=len(features))
pca_full.fit(X_scaled)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(evr)+1), evr, alpha=0.7, label='Per component')
ax.plot(range(1, len(evr)+1), cum_evr, 'o-', color='red', label='Cumulative')
ax.axhline(0.75, color='green', linestyle='--', linewidth=1, label='75% threshold')
ax.axhline(0.90, color='orange', linestyle='--', linewidth=1, label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Explained Variance')
ax.set_xticks(range(1, len(evr)+1))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/pca_explained_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Variance explained by {CFG.n_components} components: {cum_evr[CFG.n_components-1]:.3f}')
n_for_90 = next(i for i, c in enumerate(cum_evr, 1) if c >= 0.90)
print(f'Components needed for 90% variance: {n_for_90}')